In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from scipy.sparse import load_npz
import pandas as pd

X_combined_final = load_npz("../data/processed/X_combined_final.npz")
y_log = pd.read_pickle("../data/processed/y_log.pkl")


X_train, X_test, y_train_log, y_test_log = train_test_split(
    X_combined_final, y_log, test_size=0.2, random_state=42
)

def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true))**2))

In [2]:
# optuna
import optuna
from sklearn.linear_model import Ridge

def objective(trial):
    alpha = trial.suggest_float("alpha", 1e-3, 100.0, log=True)
    solver = trial.suggest_categorical("solver", ["auto", "sag", "sparse_cg"])

    model = Ridge(alpha=alpha, solver=solver, random_state=42)
    model.fit(X_train, y_train_log)

    pred_log = model.predict(X_test)
    pred = np.clip(np.expm1(pred_log), 0, None)
    y_test_orig = np.expm1(y_test_log)

    return rmsle(y_test_orig, pred)

c:\Users\mega\anaconda3\envs\ml-dev\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 실행
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

print("Best RMSLE:", study.best_value)
print("Best params:", study.best_params)

[I 2026-08-27 16:03:43,938] A new study created in memory with name: no-name-fad2f40c-8e33-46e4-b877-1eb22055d41a
[I 2026-08-27 16:04:14,317] Trial 0 finished with value: 0.4974660049137982 and parameters: {'alpha': 0.007424886687794801, 'solver': 'sparse_cg'}. Best is trial 0 with value: 0.4974660049137982.
[I 2026-08-27 16:04:41,261] Trial 1 finished with value: 0.49609853528769804 and parameters: {'alpha': 0.6422617682237381, 'solver': 'sag'}. Best is trial 1 with value: 0.49609853528769804.
[I 2026-08-27 16:04:53,126] Trial 2 finished with value: 0.5072202175305989 and parameters: {'alpha': 20.21682103154167, 'solver': 'sag'}. Best is trial 1 with value: 0.49609853528769804.
[I 2026-08-27 16:05:24,009] Trial 3 finished with value: 0.4974588326500893 and parameters: {'alpha': 0.008832475795990983, 'solver': 'auto'}. Best is trial 1 with value: 0.49609853528769804.
[I 2026-08-27 16:05:40,526] Trial 4 finished with value: 0.5012112551339384 and parameters: {'alpha': 8.513041386285328,

Best RMSLE: 0.495943344779948
Best params: {'alpha': 0.9793449961941022, 'solver': 'sag'}


In [4]:
best_params = study.best_params

final_model = Ridge(alpha=best_params["alpha"], solver=best_params["solver"], random_state=42)
final_model.fit(X_train, y_train_log)

pred_final_log = final_model.predict(X_test)
pred_final = np.clip(np.expm1(pred_final_log), 0, None)
y_test_orig = np.expm1(y_test_log)

print("Final RMSLE:", rmsle(y_test_orig, pred_final))

Final RMSLE: 0.495943344779948


In [5]:
import joblib

# 모델 저장 경로 설정 (폴더가 존재한다고 가정)
model_path = "../data/processed/best_ridge_model.pkl"

# joblib을 이용해 모델 객체 저장
joblib.dump(final_model, model_path)

print("최종 릿지 모델 저장 완료!")

최종 릿지 모델 저장 완료!
